# Train the U-Net rim segmenter on a Colab GPU

The code lives in the repository; this notebook only fetches it, unpacks the tiles and runs
`lunar_crater.training.train_unet`, once per seed. Results are written straight to Google
Drive, so they survive a disconnected session. If a session ends, run the notebook again:
finished seeds are skipped, and an unfinished seed starts over.

**Before running**

1. *Runtime → Change runtime type → T4 GPU.*
2. Make the tile zip locally and upload it to `MyDrive/lunar_crater/` in Google Drive:
   `python scripts/pack_tiles.py data/interim/tiles/v1 data/processed/lunar_tiles_v1.zip`
3. Set `BRANCH` below if the code is still on a feature branch.

With the default settings, each seed takes roughly 30 to 40 minutes on a T4.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/lunar_crater"
ZIP_NAME = "lunar_tiles_v1.zip"
ZIP_SHA256 = "da77e11db8513d901bc12d34cfccf6817af496c4ae9bbebea113aeb84265857d"
BRANCH = "master"
RUN_NAME = "ms_unet_v1"
SEEDS = [0, 1, 2]
EPOCHS = 60
LARGE_MIN_DIAM_KM = 6  # 0 trains the single-output U-Net

In [ ]:
REPO = "https://github.com/aditya-patra1011/lunar_crater_detection.git"
!git clone --quiet --depth 1 --branch {BRANCH} {REPO} /content/repo
%pip install --quiet -e /content/repo
!git -C /content/repo log --oneline -1

In [ ]:
import hashlib
import zipfile
from pathlib import Path

zip_path = Path(DRIVE_DIR) / ZIP_NAME
sha = hashlib.sha256(zip_path.read_bytes()).hexdigest()
assert sha == ZIP_SHA256, f"zip checksum {sha} does not match; re-upload it"
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall("/content/tiles")
tiles_dir = Path("/content/tiles") / ZIP_NAME.removeprefix("lunar_tiles_").removesuffix(".zip")
print(tiles_dir, len(list((tiles_dir / "tiles").glob("*.npz"))), "tiles")

In [ ]:
from pathlib import Path

import pandas as pd

options = f"--epochs {EPOCHS} --large-min-diam {LARGE_MIN_DIAM_KM}"
for seed in SEEDS:
    out_dir = Path(DRIVE_DIR) / RUN_NAME / f"seed{seed}"
    history_path = out_dir / "history.csv"
    if history_path.exists() and len(pd.read_csv(history_path)) >= EPOCHS:
        print(f"seed {seed} is already finished")
        continue
    module = "lunar_crater.training.train_unet"
    train = f"python -m {module} {tiles_dir} --out {out_dir} --seed {seed} {options}"
    !cd /content/repo && {train}

In [ ]:
for seed in SEEDS:
    history = pd.read_csv(Path(DRIVE_DIR) / RUN_NAME / f"seed{seed}" / "history.csv")
    best = history.loc[history["val_loss"].idxmin()]
    print(f"seed {seed}: best epoch {int(best['epoch'])}, val loss {best['val_loss']:.4f}")
    ax = history.plot(x="epoch", y=["train_loss", "val_loss"], figsize=(6, 3), title=f"seed {seed}")
    ax.set_ylabel("loss")

## Afterwards

Each seed has its own folder in `MyDrive/lunar_crater/ms_unet_v1/` (`seed0`, `seed1`, `seed2`).
Download each folder's `best.pt`, `config.json` and `history.csv` into the matching folder in
the local repository, for example `models/ms_unet_v1/seed0/`. They are scored there with the
evaluation harness, exactly like the baselines.